***IMPORT DE LIBRERIAS A UTILIZAR***

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import psycopg2

***CHECKEO PREVIO BOOKS TO SCRAPE***

In [3]:
try:
    # Indicamos la URL a la que realizaremos peticiones GET
    book_scrape = "https://books.toscrape.com/"
    # Donde guardaremos la respuesta que obtenemos de la página WEB
    response = requests.get(book_scrape)
    # Verificar estado de conexión 200 = exitosa
    if response.status_code == 200:
        print("Conexión exitosa...")
except (Exception,KeyboardInterrupt) as e:
    print(f"Hubo un error {e}")

Conexión exitosa...


***ITERACION POR PAGINAS PARA SCRAPEO DE INFORMACION DE LIBROS***

In [ ]:
libro_datos = []  # Creamos una lista que almacenará los datos de los libros
for page_num in range(1, 51):
    # Realizará las iteraciones del for dentro de la URL de cada página del catálogo
    books_pages = f'https://books.toscrape.com/catalogue/page-{page_num}.html'
    #peticion get a url
    response = requests.get(books_pages)

    #status code distinto a 200 es un fallo en la conexion
    if response.status_code != 200:
        print(f"Error en página {page_num}: HTTP {response.status_code}")
        continue  # salta a la siguiente página
    
    #response.content trae la informacion en bytes el encoding lo decide beatifulsoup
    soup = BeautifulSoup(response.content, 'html.parser')  # Obtener el HTML
    #encontrar todos los h3
    libros = soup.find_all('h3')

    # Iterar sobre la lista de títulos de libros
    for libro in libros:
        try:
            # Encontrar la URL vinculada al libro (libro.find)
            libro_url = libro.find('a')['href']
            # Accede a la URL de Books to Scrape + a la URL del libro
            libro_response = requests.get('https://books.toscrape.com/catalogue/' + libro_url)
            #status code distinto a 200 es un fallo en la conexion
            if libro_response.status_code != 200:
               #Dentro del try/except, raise redirige el error al except
               raise Exception(f"Error en libro: HTTP {libro_response.status_code}")  # salta al siguiente libro
        
            # Obtenemos el HTML de la página de la URL del libro
            libro_soup = BeautifulSoup(libro_response.content, "html.parser")

            # Busqueda de datos de los libros mediante etiquetas HTML
            # Encontrar el título del libro mediante la etiqueta del h1
            titulo = libro_soup.find('h1').text
            # Accede a la etiqueta ul donde encuentra la ruta de búsqueda donde se encuentra la categoría del libro
            categoria = libro_soup.find('ul', class_="breadcrumb").find_all('a')[2].text.strip()
            #Se accede al segundo elemento de de la clase star_rating donde indica la cantidad de estrellas (CSS)
            calificacion = libro_soup.find('p', class_='star-rating')['class'][1]
            precio = libro_soup.find('p', class_="price_color").text.strip()
            #agregamos toda la informacion que traemos a la lista libro_datos
            libro_datos.append([titulo, categoria, calificacion, precio])
            print(f"Titulo: {titulo}")
            print(f"Categoria: {categoria}")
            print(f"Calificacion: {calificacion}")
            print(f"Precio: {precio}")
            print("--------------")
        except (Exception,KeyboardInterrupt) as e:
            print(f"Hubo un error {e}")
            continue

***CONVERSION DE LISTA A CSV MEDIANTE PANDAS***

In [ ]:
# Crear el DataFrame indicando la lista y las columnas que tendra el dataframe
df = pd.DataFrame(libro_datos, columns=["titulo", "categoria", "calificacion", "precio"])

# Limpiar precio reemplazando "£" por ""
df["precio"] = df["precio"].str.replace("£", "").astype(float)

# Convertir calificación a número "One": 1
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df["calificacion"] = df["calificacion"].map(rating_map)



In [ ]:
# Agregar id único por fila para relacionar con los demás CSVs
df = pd.read_csv("../data/libros_scrapeados.csv")
df["autor_id"] = range(len(df))

# index=False evita que pandas agregue una columna extra con el índice numérico al CSV
df.to_csv("../data/libros_scrapeados.csv", index=False)
df

***OBTENCION DE KEY DE AUTORES MEDIANTE TITULO DE OBRA (OPEN-LIBRARY)***

In [ ]:
# Leemos el CSV para poder realizar acciones sobre los campos del CSV
datos = pd.read_csv("../data/libros_scrapeados.csv")
#convertimos la columna de titulos en una lista iterable
titulos = datos['titulo'].tolist()
#lista donde se almacenara las key de cada autor
keys = []
for titulo in titulos:
    try:
        #Se procesas los titulos cortandolo donde se encuentre los siguientes simbolos
        titulo_corto = titulo.split(":")[0].split("(")[0].split(",")[0].strip()
        #Con params requests se encarga del encoding automáticamente y pide que devuelva solo 1 resultado
        params = {"title": titulo_corto, "limit": 1}
        url_opl = "https://openlibrary.org/search.json"
        #peticion get con url de la api, parametros de filtro a la URL, si el servidor no responde en 10 segundos, lanza un error en vez de esperar
        response = requests.get(url_opl, params=params, timeout=10)
        #status code distinto a 200 es un fallo en la conexion
        if response.status_code != 200:
            raise Exception(f"HTTP {response.status_code}")
        #convierte en un diccionario de python
        data = response.json()
        # Accede al primer resultado de la lista de documentos devueltos por la API
        doc = data["docs"][0]
        # Extrae la key del primer autor del documento
        opl_id = doc["author_key"][0]

        keys.append([opl_id])
        print(keys)
    except (Exception,KeyboardInterrupt) as e:
        print(f"No encontrado: {titulo} - {e}")
        #si alguna key cae en el except se guarda como none dentro de la lista 
        keys.append([None])
        continue



***CONVERSION DE LISTA DE AUTOR_KEYS A CSV***

In [ ]:
# Convertir una lista en un DataFrame
df = pd.DataFrame(keys, columns=["autor_key"])

# Agregar id único por fila para relacionar con los demás CSVs
df["autor_id"] = range(len(df))

# index=False evita que pandas agregue una columna extra con el índice numérico al CSV
df.to_csv("autor_key.csv", index=False)
df

In [ ]:
df_keys = pd.read_csv("../data/autor_key.csv")
df_keys["autor_id"] = range(len(df_keys))
df_keys.to_csv("../data/autor_key.csv", index=False)
df_keys

***ENRIQUECIMIENTO DE DATOS DE AUTOR MEDIANTE AUTOR_KEY (OPEN-LIBRARY)***

In [ ]:
# Leemos el CSV para poder realizar acciones sobre los campos del CSV
dato_key = pd.read_csv("../data/autor_key.csv")
autor_keys = dato_key['autor_key'].tolist()

autor_datos = []

for key in autor_keys:
    # Validadcion previa. si la key es nula,guarda 4 None y saltás al siguiente
    if pd.isna(key):
        autor_datos.append([None, None, None, None])
        continue
    try:
        # URLs para obtener datos del autor y cantidad de obras usando la key de OPL
        url_autor = f"https://openlibrary.org/authors/{key}.json"
        url_works = f"https://openlibrary.org/authors/{key}/works.json"
        # Peticion GET al endpoint del autor
        response_autor = requests.get(url_autor, timeout=10)
        # Status code distinto a 200 es un fallo en la conexión
        if response_autor.status_code != 200:
            #Dentro del try/except, raise redirige el error al except
            raise Exception(f"HTTP {response_autor.status_code}")
        # Petición GET al endpoint de las obras del autor
        response_works = requests.get(url_works, timeout=10)
    
        if response_works.status_code != 200:
            raise Exception(f"HTTP {response_works.status_code}")
        # Convierte las respuestas en diccionarios de Python
        data_works = response_works.json()
        data_autor = response_autor.json()

        # Extraer datos del autor, none en caso de que el capo no se encuentre en el JSON
        birth_date = data_autor.get("birth_date", None)
        nombre = data_autor.get("personal_name", None)
        #created es un diccionario anidado dentro de el JSON 
        fecha_creacion = data_autor.get("created", {}).get("value", None)
        #size es el total de obras del autor
        total_obras = data_works.get("size", None)

        # Procesar año de nacimiento
        anho_nacimiento = None
        #OPL no tiene formato estandar para birth_date
        if birth_date:
            #resta -3 para siempre poder encontrar 4 caracteres 
            for i in range(len(birth_date) - 3):
                #cada iteracion toma una ventana de 4 caracteres
                parte = birth_date[i:i+4]
                #cuando encuentra 4 caracteres consecutivos que son numeros, los convierte a entero y para el loop
                if parte.isdigit():
                    anho_nacimiento = int(parte)
                    break
        #Se agregan los datos a la lista de autor_datos
        autor_datos.append([anho_nacimiento, nombre, fecha_creacion, total_obras])
        # Pausa entre peticiones para no sobrecargar el servidor
        time.sleep(0.5) 
        print(f"Nombre: {nombre}")
        print(f"Año: {anho_nacimiento}")
        print(f"Creación: {fecha_creacion}")
        print(f"Cantidad Obras: {total_obras}")
        print("--------------")
    except (Exception,KeyboardInterrupt) as e:
        print(f"Error: {key} - {e}")
        #si algo falla en el proceso se guarda la fila con los none y se pasa a la siguiente
        autor_datos.append([None, None, None, None])
        continue



***CONVERSION LISTA DE DATOS DE AUTOR A CSV MEDIANTE PANDAS***

In [ ]:
# Convertir una lista en un DataFrame
df = pd.DataFrame(autor_datos, columns=["anho_nacimiento", "nombre", "fecha_creacion", "total_obras"])

# index=False evita que pandas agregue una columna extra con el índice numérico al CSV
df.to_csv("autor_datos.csv", index=False)
df

In [ ]:
df = pd.read_csv("../data/autor_datos.csv")
# errors="coerce" convierte valores inválidos o nulos en NaN en vez de lanzar un error
df["anho_nacimiento"] = pd.to_numeric(df["anho_nacimiento"], errors="coerce").astype("Int64")
# astype("Int64") convierte a entero respetando los NaN, int normal no acepta NaN
df["total_obras"] = pd.to_numeric(df["total_obras"], errors="coerce").astype("Int64")

df["autor_id"] = range(len(df))
df.to_csv("../data/autor_datos.csv", index=False)
df

***ENRIQUECIMIENTO DE NACIONALIDAD DE AUTORES MEDIANTE NOMBRE DE AUTOR (WIKIPEDIA)***

In [ ]:
# Leemos el CSV para poder realizar acciones sobre los campos del CSV
dato_nombre = pd.read_csv("../data/autor_datos.csv")
# Convertimos la columna de nombres en una lista iterable
autor_nombre = dato_nombre['nombre'].tolist()

nacionalidades = []
for nombre in autor_nombre:
    # Validación previa. Si el nombre es nulo, guarda None y salta al siguiente
    if pd.isna(nombre):
        nacionalidades.append(None)
        continue
    try:
        # Headers para identificar el cliente ante la API de Wikipedia
        headers = {
            "User-Agent": "books-scrape-challenge/1.0 (hugo@email.com)"
        }
        # Limpiar nombre: quitar espacios y puntos al final
        nombre_limpio = nombre.strip().rstrip(".")
        # Reemplaza espacios por guiones bajos para construir la URL de Wikipedia
        url_wiki = f"https://en.wikipedia.org/api/rest_v1/page/summary/{nombre_limpio.replace(' ', '_')}"
        # Petición GET al endpoint de Wikipedia
        response_wiki = requests.get(url_wiki, headers=headers, timeout=10)
        # Status code distinto a 200 es un fallo en la conexión
        if response_wiki.status_code != 200:
            raise Exception(f"HTTP {response_wiki.status_code}")
        # Convierte la respuesta en un diccionario de Python
        data_wiki = response_wiki.json()
        # Extrae la descripción del autor, None si no se encuentra
        description = data_wiki.get("description", None)

        # La descripción suele empezar con la nacionalidad, ej: "American novelist"
        if description and description != "Topics referred to by the same term":
            nacionalidad = description.split()[0]
        else:
            nacionalidad = None
        nacionalidades.append(nacionalidad)
        # Pausa entre peticiones para no sobrecargar el servidor
        time.sleep(0.3)
        print(nacionalidades)
    except (Exception, KeyboardInterrupt) as e:
        print(f"Error: {nombre} - {e}")
        # Si algo falla en el proceso se guarda None y se pasa al siguiente
        nacionalidades.append(None)
        continue

***CONVERSION LISTA DE NACIONALIDADES A CSV MEDIANTE PANDAS***

In [ ]:
# Convertir una lista en un DataFrame
df = pd.DataFrame(nacionalidades, columns=["nacionalidad"])

# index=False evita que pandas agregue una columna extra con el índice numérico al CSV
df.to_csv("autor_nacionalidad.csv", index=False)
df

In [ ]:
df = pd.read_csv("../data/autor_nacionalidad.csv")
df["autor_id"] = range(len(df))
df.to_csv("../data/autor_nacionalidad.csv", index=False)
df

***VERIFICAMOS CANTIDAD DE INFORMACION ALMACENADA EN LOS CSV***

In [12]:
# Leer los CSVs a través de la ruta de archivos
df_libros = pd.read_csv("../data/libros_scrapeados.csv")
df_keys = pd.read_csv("../data/autor_key.csv")
df_autores = pd.read_csv("../data/autor_datos.csv")
df_nacion = pd.read_csv("../data/autor_nacionalidad.csv")
# Imprimimos la cantidad de filas de cada CSV para ver cantidad de información
print(len(df_libros), len(df_keys), len(df_autores), len(df_nacion))

1000 1000 1000 1000


In [ ]:
df = pd.read_csv("../data/autor_datos.csv")
print(df["fecha_creacion"].head(10))

***CONEXION A BASE DE DATOS***

In [40]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    user="postgres",
    password="12345",
    database="ConsultaMortal"
)
cursor = conn.cursor()

***CREACION DE TABLAS STAGING***

In [41]:
staging_ddl = """
DROP TABLE IF EXISTS staging_books, staging_authors, staging_keys, staging_nationalities CASCADE;

CREATE TABLE staging_books (
    titulo       TEXT,
    categoria    TEXT,
    calificacion TEXT,
    precio       TEXT,
    autor_id     INTEGER
);

CREATE TABLE staging_authors (
    anho_nacimiento TEXT,
    nombre          TEXT,
    fecha_creacion  TEXT,
    total_obras     TEXT,
    autor_id        INTEGER
);

CREATE TABLE staging_keys (
    autor_key TEXT,
    autor_id  INTEGER
);

CREATE TABLE staging_nationalities (
    nacionalidad TEXT,
    autor_id     INTEGER
);
"""

cursor.execute(staging_ddl)
conn.commit()
print("Tablas staging creadas")

Tablas staging creadas


***CARGA DE DATOS A TABLAS STAGING***

In [42]:
# Cargar libros
with open("../data/libros_scrapeados.csv", "r", encoding="utf-8") as f:
    cursor.copy_expert("""
        COPY staging_books (titulo, categoria, calificacion, precio, autor_id)
        FROM STDIN WITH (FORMAT csv, HEADER true, DELIMITER ',')
    """, f)

# Cargar autores
with open("../data/autor_datos.csv", "r", encoding="utf-8") as f:
    cursor.copy_expert("""
        COPY staging_authors (anho_nacimiento, nombre, fecha_creacion, total_obras, autor_id)
        FROM STDIN WITH (FORMAT csv, HEADER true, DELIMITER ',')
    """, f)

# Cargar keys
with open("../data/autor_key.csv", "r", encoding="utf-8") as f:
    cursor.copy_expert("""
        COPY staging_keys (autor_key, autor_id)
        FROM STDIN WITH (FORMAT csv, HEADER true, DELIMITER ',')
    """, f)

# Cargar nacionalidades
with open("../data/autor_nacionalidad.csv", "r", encoding="utf-8") as f:
    cursor.copy_expert("""
        COPY staging_nationalities (nacionalidad, autor_id)
        FROM STDIN WITH (FORMAT csv, HEADER true, DELIMITER ',')
    """, f)

conn.commit()
print("CSVs cargados a staging")

CSVs cargados a staging


***CREACION DE SCHEMA DEFINITIVO PARA LA BASE DE DATOS***


In [63]:
schema_ddl = """
DROP TABLE IF EXISTS book_author CASCADE;
DROP TABLE IF EXISTS books CASCADE;
DROP TABLE IF EXISTS authors CASCADE;
DROP TABLE IF EXISTS categories CASCADE;

CREATE TABLE categories (
    id_category SERIAL PRIMARY KEY,
    category_name VARCHAR(80) NOT NULL UNIQUE
);

CREATE TABLE authors (
    id_author        SERIAL PRIMARY KEY,
    author_name      VARCHAR(100),
    birth_year       INTEGER,
    country          VARCHAR(100),
    external_api_id  VARCHAR(100),
    total_known_works INTEGER,
    api_source       VARCHAR(100),
    created_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE books (
    id_book     SERIAL PRIMARY KEY,
    id_category INTEGER NOT NULL,
    title       VARCHAR(500),
    price       NUMERIC(12,2),
    rating      SMALLINT,
    CONSTRAINT fk_category FOREIGN KEY (id_category)
        REFERENCES categories(id_category)
);

CREATE TABLE book_author (
    id_book   INTEGER NOT NULL,
    id_author INTEGER NOT NULL,
    PRIMARY KEY (id_book, id_author),
    CONSTRAINT fk_book   FOREIGN KEY (id_book)   REFERENCES books(id_book),
    CONSTRAINT fk_author FOREIGN KEY (id_author) REFERENCES authors(id_author)
);

CREATE INDEX idx_books_category ON books(id_category);
CREATE INDEX idx_books_rating   ON books(rating);
CREATE INDEX idx_books_price    ON books(price);
CREATE INDEX idx_authors_name   ON authors(author_name);
CREATE INDEX idx_book_author_book ON book_author(id_book);
CREATE INDEX idx_book_author_auth ON book_author(id_author);
"""

cursor.execute(schema_ddl)
conn.commit()
print("Schema final creado")

Schema final creado


In [80]:
conn.rollback()

transform_load = """
-- 1. Categorías únicas
-- DISTINCT evita duplicados, TRIM limpia espacios, ORDER BY las inserta alfabéticamente
INSERT INTO categories (category_name)
SELECT DISTINCT TRIM(categoria)
FROM staging_books
WHERE categoria IS NOT NULL
ORDER BY TRIM(categoria);

-- 2. Autores únicos
-- NULLIF convierte strings vacíos en NULL
-- autor_id es un id generado en pandas que alinea las 3 tablas de staging por posicion
-- ya que fueron cargadas en el mismo orden desde los CSVs
INSERT INTO authors (author_name, birth_year, country, external_api_id, total_known_works, api_source, created_at)
SELECT DISTINCT
    TRIM(sa.nombre),
    NULLIF(SPLIT_PART(TRIM(sa.anho_nacimiento), '.', 1), '')::INTEGER,
    NULLIF(TRIM(sn.nacionalidad), ''),
    NULLIF(TRIM(sk.autor_key), ''),
    NULLIF(SPLIT_PART(TRIM(sa.total_obras), '.', 1), '')::INTEGER,
    'OpenLibrary + Wikipedia',
    sa.fecha_creacion::TIMESTAMP
FROM staging_authors sa
JOIN staging_keys sk          ON sa.autor_id = sk.autor_id
JOIN staging_nationalities sn ON sa.autor_id = sn.autor_id
WHERE sk.autor_key IS NOT NULL;

-- 3. Libros
-- JOIN a categories para obtener el id_category generado en el paso 1
INSERT INTO books (id_category, title, price, rating)
SELECT
    c.id_category,
    TRIM(sb.titulo),
    sb.precio::NUMERIC(12,2),
    sb.calificacion::SMALLINT
FROM staging_books sb
JOIN categories c ON TRIM(sb.categoria) = c.category_name;

-- 4. Relacion libro-autor
INSERT INTO book_author (id_book, id_author)
SELECT DISTINCT
    b.id_book,
    a.id_author
FROM staging_books sb
JOIN staging_keys sk ON sb.autor_id = sk.autor_id
JOIN books b         ON TRIM(sb.titulo) = b.title
JOIN authors a       ON TRIM(sk.autor_key) = a.external_api_id;
"""

cursor.execute(transform_load)
conn.commit()
print("Datos cargados correctamente")

Datos cargados correctamente


In [78]:
conn.rollback()

cursor.execute("""
    TRUNCATE TABLE book_author, books, authors, categories RESTART IDENTITY CASCADE;
""")
conn.commit()
print("Tablas limpiadas")

Tablas limpiadas


***VERIFICAMOS QUE LAS TABLAS FUERON CARGADAS CORRECTAMENTE***

In [81]:
cursor.execute("SELECT COUNT(*) FROM categories")
print("Categorías:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM authors")
print("Autores:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM books")
print("Libros:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM book_author")
print("Relaciones book_author:", cursor.fetchone()[0])

Categorías: 50
Autores: 808
Libros: 1000
Relaciones book_author: 972


***5 CONSULTAS A LA BASE DE DATOS***

In [ ]:
# CONSULTA 1: Libros con rating > 3 y precio < £10
# Filtra libros bien valorados y económicos
cursor.execute("""
    SELECT b.title, b.price, b.rating, c.category_name
    FROM books b
    JOIN categories c ON b.id_category = c.id_category
    WHERE b.rating > 3 AND b.price < 10
    ORDER BY b.rating DESC, b.price ASC;
""")
df1 = pd.DataFrame(cursor.fetchall(), columns=["titulo", "precio", "rating", "categoria"])
print("Consulta 1: Libros con rating > 3 y precio < £10")
print(df1)

# CONSULTA 2: Autor con peor promedio de rating (mínimo 5 libros, usando subconsulta)
# Identifica autores con producción considerable pero mala recepción
cursor.execute("""
    SELECT a.author_name, sub.total_libros, ROUND(sub.promedio_rating, 2) AS promedio_rating
    FROM authors a
    JOIN (
        SELECT ba.id_author, COUNT(b.id_book) AS total_libros, AVG(b.rating) AS promedio_rating
        FROM book_author ba
        JOIN books b ON ba.id_book = b.id_book
        GROUP BY ba.id_author
        HAVING COUNT(b.id_book) >= 5
    ) sub ON a.id_author = sub.id_author
    ORDER BY sub.promedio_rating ASC
    LIMIT 10;
""")
df2 = pd.DataFrame(cursor.fetchall(), columns=["autor", "total_libros", "promedio_rating"])
print("\nConsulta 2: Autor con peor promedio de rating (mínimo 5 libros, con subconsulta)")
print(df2)

# CONSULTA 3: Categoría con mayor precio promedio
# Revela qué géneros son más caros en el mercado
cursor.execute("""
    SELECT c.category_name, ROUND(AVG(b.price), 2) AS precio_promedio
    FROM categories c
    JOIN books b ON c.id_category = b.id_category
    GROUP BY c.category_name
    ORDER BY precio_promedio DESC
    LIMIT 10;
""")
df3 = pd.DataFrame(cursor.fetchall(), columns=["categoria", "precio_promedio"])
print("\nConsulta 3: Categoría con mayor precio promedio")
print(df3)

# CONSULTA 4: Top 5 autores con más libros (usando función de ventana ROW_NUMBER)
cursor.execute("""
    SELECT author_name, total_libros
    FROM (
        SELECT a.author_name, COUNT(b.id_book) AS total_libros,
               ROW_NUMBER() OVER (ORDER BY COUNT(b.id_book) DESC) AS rn
        FROM authors a
        JOIN book_author ba ON a.id_author = ba.id_author
        JOIN books b ON ba.id_book = b.id_book
        GROUP BY a.author_name
    ) ranked
    WHERE rn <= 5;
""")
df4 = pd.DataFrame(cursor.fetchall(), columns=["autor", "total_libros"])
print("\nConsulta 4: Top 5 autores con más libros (con función de ventana)")
print(df4)

# CONSULTA 5 (OBLIGATORIA): País que produce más libros con rating > 3
# Requiere JOIN completo books → book_author → authors → GROUP BY country
# Sin la API esta consulta no existiría

cursor.execute("""
    SELECT a.country, COUNT(b.id_book) AS total_libros
    FROM books b
    JOIN book_author ba ON b.id_book = ba.id_book
    JOIN authors a ON ba.id_author = a.id_author
    WHERE b.rating > 3 AND a.country IS NOT NULL
    GROUP BY a.country
    ORDER BY total_libros DESC
    LIMIT 10;
""")
df5 = pd.DataFrame(cursor.fetchall(), columns=["pais", "total_libros"])
print("\nConsulta 5 (Obligatoria): País con más libros de rating > 3")
print(df5)

In [ ]:
# INDEXACIÓN Y PERFORMANCE

# Eliminar índice para partir desde cero
cursor.execute("DROP INDEX IF EXISTS idx_books_price;")
conn.commit()

#SIN índice
cursor.execute("EXPLAIN ANALYZE SELECT title, price FROM books WHERE price > 50;")
print("SIN INDICE:")
for row in cursor.fetchall():
    print(row[0])

# Crear índice
cursor.execute("CREATE INDEX idx_books_price ON books(price);")
conn.commit()

#CON índice 
cursor.execute("EXPLAIN ANALYZE SELECT title, price FROM books WHERE price > 50;")
print("CON INDICE:")
for row in cursor.fetchall():
    print(row[0])